# Fine-tune CLAP with LoRA on MusicCaps

Fine-tunes the text encoder of the pre-trained CLAP music model using LoRA adapters. The audio encoder stays frozen throughout.

Downloads MusicCaps clips, pre-computes audio embeddings, then trains with symmetric contrastive loss. LoRA weights are saved to `checkpoints/clap_lora/`.

Run `compute_embeddings.ipynb` first so the base checkpoint is already on disk.

In [ ]:
import sys
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split

if "" not in sys.path:
    sys.path.insert(0, "")

import utils
import fma_utils

MUSICCAPS_DIR = Path("data/musiccaps")
CAPTIONS_CSV = MUSICCAPS_DIR / "captions.csv"
EMB_PATH = Path("data/musiccaps_embeddings.npz")
LORA_CKPT_DIR = Path("checkpoints/clap_lora")

CHECKPOINT_PATH = utils.DEFAULT_CHECKPOINT_PATH
CHECKPOINT_URL = utils.DEFAULT_CHECKPOINT_URL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

BATCH_SIZE = 16
EPOCHS = 3
LR = 1e-4
TEST_SPLIT = 0.15
DL_WORKERS = 4
YT_BROWSER = "chrome"
YT_JS_RUNTIME = "node"

MUSICCAPS_DIR.mkdir(parents=True, exist_ok=True)
LORA_CKPT_DIR.mkdir(parents=True, exist_ok=True)

FMA_METADATA_DIR = Path("data/fma_metadata")
FMA_EMB_PATH = Path("data/embeddings.npz")
FMA_SIZE = "small"

print(f"Device : {DEVICE}")
print(f"LoRA   : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Train  : batch={BATCH_SIZE}, epochs={EPOCHS}, lr={LR}")

## MusicCaps metadata

In [ ]:
from datasets import load_dataset

if CAPTIONS_CSV.exists():
    captions_df = pd.read_csv(CAPTIONS_CSV)
    print(f"Captions already on disk — loaded {len(captions_df)} rows from {CAPTIONS_CSV}")
else:
    ds = load_dataset("google/MusicCaps", split="train")
    print(f"MusicCaps rows: {len(ds)}")
    captions_df = ds.to_pandas()[["ytid", "start_s", "end_s", "caption"]]
    captions_df.to_csv(CAPTIONS_CSV, index=False)
    print(f"Captions saved → {CAPTIONS_CSV}  ({len(captions_df)} rows)")

captions_df.head(3)

## Download audio clips

Downloads clips as WAV trimmed to `[start_s, end_s]`. Already-downloaded files are skipped. Results are saved to `download_status.json`.

In [ ]:
import yt_dlp
from yt_dlp.utils import download_range_func

STATUS_FILE = MUSICCAPS_DIR / "download_status.json"

# Error substrings that mean the video will never be downloadable — don't retry
_PERMANENT_PATTERNS = (
    "private",
    "terminated",
    "copyright",
    "blocked",
    "no longer available",
    "this video is not available",
    "video unavailable",
    "has been removed",
    "not made this video available",
    "processing this video"
)

def _is_permanent(msg: str) -> bool:
    low = msg.lower()
    return any(p in low for p in _PERMANENT_PATTERNS)


class _YtLogger:
    """Captures yt-dlp log lines so we can show them on failure."""
    def __init__(self):
        self.lines: list[str] = []
    def debug(self, msg: str) -> None:
        pass
    def warning(self, msg: str) -> None:
        self.lines.append(f"WARNING: {msg}")
    def error(self, msg: str) -> None:
        self.lines.append(f"ERROR: {msg}")


# Status values:
#   True        — downloaded successfully
#   False       — transient failure (network, bot-detection) → will retry
#   "permanent" — video gone / private / geo-blocked       → will never retry
def download_clip(ytid: str, start_s: float, end_s: float):
    """Download a single MusicCaps clip. Returns (ytid, status, msg)."""
    out_path = MUSICCAPS_DIR / f"{ytid}.wav"
    if out_path.exists():
        return ytid, True, "already exists"

    logger = _YtLogger()
    ydl_opts = {
        "quiet": True,
        "logger": logger,
        "format": "bestaudio/best",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "outtmpl": str(out_path.with_suffix("")),
        "download_ranges": download_range_func(None, [(start_s, end_s)]),
        "force_keyframes_at_cuts": True,
        "cookiesfrombrowser": (YT_BROWSER,),
        "js_runtimes": {YT_JS_RUNTIME: {}},
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([f"https://www.youtube.com/watch?v={ytid}"])
        return ytid, True, "downloaded"
    except Exception as exc:
        msg = "\n".join(logger.lines) or str(exc)
        verdict = "permanent" if _is_permanent(msg) else False
        return ytid, verdict, msg


# Load previous results — tri-state: True / False / "permanent"
status: dict = {}
if STATUS_FILE.exists():
    with open(STATUS_FILE) as f:
        status = json.load(f)
    n_ok   = sum(1 for v in status.values() if v is True or v == True)
    n_perm = sum(1 for v in status.values() if v == "permanent")
    print(f"Loaded {len(status)} clips: {n_ok} OK, {n_perm} permanently unavailable")

all_rows = captions_df[["ytid", "start_s", "end_s"]].to_dict("records")
# Skip successes AND permanent failures — only retry transient failures
pending_rows = [r for r in all_rows if not status.get(r["ytid"], False)]
print(f"Clips to download: {len(pending_rows)} "
      f"(skipping {len(all_rows) - len(pending_rows)} already done/permanent)")

done = 0
with ThreadPoolExecutor(max_workers=DL_WORKERS) as pool:
    futures = {
        pool.submit(download_clip, r["ytid"], r["start_s"], r["end_s"]): r["ytid"]
        for r in pending_rows
    }
    for fut in as_completed(futures):
        ytid, ok, msg = fut.result()
        status[ytid] = ok
        done += 1
        if ok is True:
            if done % 100 == 0:
                print(f"[{done}/{len(pending_rows)}] OK      {ytid}")
        else:
            tag = "PERM" if ok == "permanent" else "FAIL"
            print(f"[{done}/{len(pending_rows)}] {tag}  {ytid}")
            for line in msg.splitlines():
                print(f"  {line}")

with open(STATUS_FILE, "w") as f:
    json.dump(status, f, indent=2)

n_ok   = sum(1 for v in status.values() if v is True or v == True)
n_perm = sum(1 for v in status.values() if v == "permanent")
n_fail = len(status) - n_ok - n_perm
print(f"\nTotal: {n_ok} OK  |  {n_perm} permanent  |  {n_fail} transient failures")

## Pre-compute audio embeddings

Run the frozen CLAP audio encoder over all clips and cache the results. Only needs to run once since the audio encoder doesn't change during training.

In [ ]:
if EMB_PATH.exists():
    print(f"Audio embeddings already on disk ({EMB_PATH}) — skipping pre-compute.")
    print("Delete the file and re-run this cell only if you've added new clips.")
else:
    # Scan for WAVs directly — no dependency on the `status` variable so this
    # cell is safe to run after a kernel restart
    wav_paths = sorted(str(p) for p in MUSICCAPS_DIR.glob("*.wav"))
    print(f"WAV files to embed: {len(wav_paths)}")

    model = utils.load_clap_model(CHECKPOINT_PATH, CHECKPOINT_URL, device=DEVICE)
    model.eval()

    EMBED_BATCH = 16
    all_embs: list[np.ndarray] = []
    all_ytids: list[str] = []

    with torch.no_grad():
        for start in range(0, len(wav_paths), EMBED_BATCH):
            batch_paths = wav_paths[start : start + EMBED_BATCH]
            embs = model.get_audio_embedding_from_filelist(batch_paths, use_tensor=False)
            embs = utils.l2_normalize(np.asarray(embs, dtype=np.float32))
            all_embs.append(embs)
            all_ytids.extend(Path(p).stem for p in batch_paths)
            if (start // EMBED_BATCH + 1) % 20 == 0:
                print(f"  embedded {start + len(batch_paths)}/{len(wav_paths)}")

    audio_embs = np.vstack(all_embs)
    utils.save_embeddings(EMB_PATH, all_ytids, audio_embs)
    print(f"Audio embeddings shape: {audio_embs.shape}")

## Dataset and DataLoaders

In [ ]:
full_dataset = utils.MusicCapsDataset(EMB_PATH, CAPTIONS_CSV)

n_test  = max(1, int(len(full_dataset) * TEST_SPLIT))
n_train = len(full_dataset) - n_test
train_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_test],
    generator=torch.Generator().manual_seed(42),
)

# Expose raw indices so we can pass them to the cosine-sim evaluator
train_indices = list(train_ds.indices)
test_indices  = list(test_ds.indices)

# text captions are strings → custom collate returns (Tensor[B, D], list[str])
def collate_fn(batch):
    embs, captions = zip(*batch)
    return torch.stack(embs), list(captions)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == "cuda"),
    drop_last=True,   # uniform batch size for contrastive loss
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == "cuda"),
    drop_last=False,
)

print(f"Train : {n_train} samples  ({len(train_loader)} batches)")
print(f"Test  : {n_test} samples   ({len(test_loader)} batches)")

## Baseline cosine similarity (before fine-tuning)

In [ ]:
# Load vanilla CLAP (no LoRA) for the baseline measurement
baseline_model = utils.load_clap_model(CHECKPOINT_PATH, CHECKPOINT_URL, device=DEVICE)

baseline_cosine_sim = utils.eval_audio_text_cosine_similarity(
    baseline_model,
    full_dataset,
    test_indices,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)
print(f"Baseline avg cosine similarity on test set : {baseline_cosine_sim:.4f}")

## FMA genre alignment — baseline

For each top-level FMA genre, compute cosine similarity between the genre's text embedding and the mean audio embedding of its tracks.

In [ ]:
genre_map = fma_utils.load_fma_genre_map(FMA_METADATA_DIR, fma_size=FMA_SIZE)
fma_paths, fma_embs = utils.load_embeddings(FMA_EMB_PATH)

baseline_genre_sims = fma_utils.compute_genre_text_audio_alignment(
    baseline_model, genre_map, fma_paths, fma_embs, DEVICE
)

print(f"FMA {FMA_SIZE.capitalize()} — Baseline genre text↔audio alignment")
print(f"{'Genre':<20} {'Cosine Sim':>12}")
print("-" * 34)
for genre, sim in sorted(baseline_genre_sims.items(), key=lambda x: -x[1]):
    print(f"{genre:<20} {sim:>12.4f}")

del baseline_model  # free VRAM before LoRA model is loaded

## FMA intra-genre cohesion

Mean pairwise cosine similarity within each genre cluster. This is a fixed audio-space metric — it doesn't change with LoRA fine-tuning since the audio encoder stays frozen.

In [ ]:
intra_genre_sims = fma_utils.compute_intra_genre_cosine_similarity(
    genre_map, fma_paths, fma_embs
)

print(f"FMA {FMA_SIZE} Intra-genre audio cohesion (mean pairwise cosine sim)")
print(f"{'Genre':<20} {'Mean cos sim':>14} {'nbtracks':>10}")
print("-" * 46)
mean_sim = 0
for genre, sim in sorted(intra_genre_sims.items(), key=lambda x: -x[1]):
    n_tracks = sum(
        1 for p in fma_paths
        if (tid := fma_utils._track_id_from_path(p)) is not None
        and genre_map.get(tid) == genre
    )
    print(f"{genre:<20} {sim:>14.4f} {n_tracks:>10}")
    mean_sim += sim
print(f"{'':<20} {mean_sim / len(intra_genre_sims):>14.4f} {sum(intra_genre_sims.values()):>10}")


## Inject LoRA into the text encoder

In [ ]:
# Re-load a fresh model so we start from the original checkpoint
model = utils.load_clap_model(CHECKPOINT_PATH, CHECKPOINT_URL, device=DEVICE)

utils.inject_lora_text_encoder(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["query", "value"],
)

# Only pass trainable params to the optimizer
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(train_loader)
)

## Training loop

In [ ]:
import torch.nn.functional as F

def encode_text_with_grad(captions: list[str]) -> torch.Tensor:
    text_input = model.tokenizer(captions)
    text_embs = model.model.encode_text(text_input, device=DEVICE)
    return F.normalize(text_embs, dim=-1)


def run_train_epoch(loader) -> float:
    model.train()
    total_loss = 0.0
    for audio_embs, captions in loader:
        audio_embs = audio_embs.to(DEVICE)
        text_embs = encode_text_with_grad(captions)
        logit_scale = model.model.logit_scale_a
        loss = utils.symmetric_contrastive_loss(audio_embs, text_embs, logit_scale)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1.0
        )
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def run_val_epoch(loader) -> float:
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for audio_embs, captions in loader:
            audio_embs = audio_embs.to(DEVICE)
            text_input = model.tokenizer(captions)
            text_embs = model.model.encode_text(text_input, device=DEVICE)
            text_embs = F.normalize(text_embs, dim=-1)
            logit_scale = model.model.logit_scale_a
            loss = utils.symmetric_contrastive_loss(audio_embs, text_embs, logit_scale)
            total_loss += loss.item()
    return total_loss / len(loader)


history: list[dict] = []

for epoch in range(1, EPOCHS + 1):
    train_loss = run_train_epoch(train_loader)
    val_loss   = run_val_epoch(test_loader)
    history.append({"train": train_loss, "val": val_loss})
    print(f"Epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

print("\nTraining complete.")

## Save LoRA checkpoint

In [ ]:
model.model.text_branch.save_pretrained(str(LORA_CKPT_DIR))
print(f"LoRA adapter weights saved → {LORA_CKPT_DIR}")

meta = {
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "loss_per_epoch": history,
}
with open(LORA_CKPT_DIR / "training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"Training metadata saved → {LORA_CKPT_DIR / 'training_meta.json'}")

## Post-training evaluation

In [ ]:
finetuned_cosine_sim = utils.eval_audio_text_cosine_similarity(
    model,
    full_dataset,
    test_indices,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

delta = finetuned_cosine_sim - baseline_cosine_sim
direction = "improvement" if delta > 0 else "regression"

print(f"Baseline avg cosine similarity  : {baseline_cosine_sim:.4f}")
print(f"Fine-tuned avg cosine similarity: {finetuned_cosine_sim:.4f}")
print(f"Delta                           : {delta:+.4f}  ({direction})")

# Persist the comparison alongside the training meta
meta_path = LORA_CKPT_DIR / "training_meta.json"
with open(meta_path) as f:
    meta = json.load(f)
meta["eval"] = {
    "test_n": len(test_indices),
    "baseline_cosine_sim": baseline_cosine_sim,
    "finetuned_cosine_sim": finetuned_cosine_sim,
    "delta": delta,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"\nEval results appended to {meta_path}")

## FMA genre alignment — after fine-tuning

In [ ]:
finetuned_genre_sims = fma_utils.compute_genre_text_audio_alignment(
    model, genre_map, fma_paths, fma_embs, DEVICE
)

print(f"FMA {FMA_SIZE.capitalize()} — Genre text↔audio alignment: baseline vs fine-tuned")
print(f"{'Genre':<20} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 54)
all_genres = sorted(set(baseline_genre_sims) | set(finetuned_genre_sims))
for genre in all_genres:
    b = baseline_genre_sims.get(genre, float("nan"))
    ft = finetuned_genre_sims.get(genre, float("nan"))
    delta = ft - b
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "—")
    print(f"{genre:<20} {b:>10.4f} {ft:>12.4f} {arrow}{abs(delta):>7.4f}")

## Phase 2 — Fine-tune the audio encoder (last Swin stage)

Phase 1 trained only the **text encoder** (LoRA) while the audio encoder was
fully frozen.  Phase 2 keeps the LoRA weights frozen and unfreezes the **last
Swin Transformer stage** of the audio encoder together with the audio
transform and projection heads.

This lets the audio encoder specialise its high-level representations to the
music domain without disturbing the stable lower-level features, and directly
improves intra-genre cluster cohesion.

Key differences from Phase 1:
- Raw WAV files are loaded on-the-fly; pre-computed embeddings are not used.
- A much smaller LR (5 × 10⁻⁶) to avoid catastrophic forgetting.
- Only 2 epochs — the audio encoder is already well pre-trained.

In [ ]:

LR_AUDIO     = 5e-6  
EPOCHS_AUDIO = 2
AUDIO_STAGES = 1    
AUDIO_CKPT_DIR = Path("checkpoints/clap_audio_phase2")
AUDIO_CKPT_DIR.mkdir(parents=True, exist_ok=True)

utils.unfreeze_audio_encoder(model, top_n_stages=AUDIO_STAGES)


raw_dataset = utils.MusicCapsRawDataset(MUSICCAPS_DIR, CAPTIONS_CSV)
n_test_raw  = max(1, int(len(raw_dataset) * TEST_SPLIT))
n_train_raw = len(raw_dataset) - n_test_raw
raw_train_ds, raw_test_ds = random_split(
    raw_dataset,
    [n_train_raw, n_test_raw],
    generator=torch.Generator().manual_seed(42),
)

RAW_BATCH = max(4, BATCH_SIZE // 2)
raw_train_loader = DataLoader(raw_train_ds, batch_size=RAW_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
raw_test_loader  = DataLoader(raw_test_ds,  batch_size=RAW_BATCH, shuffle=False, num_workers=2, pin_memory=True)

audio_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_AUDIO,
    weight_decay=1e-4,
)
audio_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    audio_optimizer, T_max=EPOCHS_AUDIO * len(raw_train_loader)
)

print(f"\nPhase-2 dataset : {n_train_raw} train / {n_test_raw} test clips")
print(f"Phase-2 config  : LR={LR_AUDIO}, epochs={EPOCHS_AUDIO}, Swin stages={AUDIO_STAGES}, batch={RAW_BATCH}")

## Phase 2 — Training loop

In [ ]:
def encode_audio_live(waveforms: torch.Tensor) -> torch.Tensor:
    """Run raw waveforms through the (partially unfrozen) audio encoder."""
    audio_embs = model.model.encode_audio({"waveform": waveforms}, device=DEVICE)["embedding"]
    audio_embs = model.model.audio_projection(audio_embs.float())
    return F.normalize(audio_embs, dim=-1)


def encode_text_frozen(captions: list[str]) -> torch.Tensor:
    """Encode captions with the frozen LoRA text encoder (no gradient)."""
    with torch.no_grad():
        text_input = model.tokenizer(captions)
        text_embs  = model.model.encode_text(text_input, device=DEVICE)
    return F.normalize(text_embs.float(), dim=-1)


def run_audio_train_epoch(loader) -> float:
    model.train()
    model.model.text_branch.eval()  # disable LoRA dropout during Phase-2
    total_loss = 0.0
    for waveforms, captions in loader:
        waveforms  = waveforms.to(DEVICE)
        audio_embs = encode_audio_live(waveforms)
        text_embs  = encode_text_frozen(captions)
        logit_scale = model.model.logit_scale_a
        loss = utils.symmetric_contrastive_loss(audio_embs, text_embs, logit_scale)
        audio_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1.0
        )
        audio_optimizer.step()
        audio_scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def run_audio_val_epoch(loader) -> float:
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for waveforms, captions in loader:
            waveforms  = waveforms.to(DEVICE)
            audio_embs = encode_audio_live(waveforms)
            text_embs  = encode_text_frozen(captions)
            logit_scale = model.model.logit_scale_a
            loss = utils.symmetric_contrastive_loss(audio_embs, text_embs, logit_scale)
            total_loss += loss.item()
    return total_loss / len(loader)


audio_history: list[dict] = []
for epoch in range(1, EPOCHS_AUDIO + 1):
    train_loss = run_audio_train_epoch(raw_train_loader)
    val_loss   = run_audio_val_epoch(raw_test_loader)
    audio_history.append({"epoch": epoch, "train": train_loss, "val": val_loss})
    print(f"Epoch {epoch}/{EPOCHS_AUDIO}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

print("\nPhase-2 training complete.")

## Phase 2 — Save checkpoint

Save the updated audio branch, transform, and projection weights so the
Phase-2 model can be reloaded independently of LoRA.

In [ ]:
audio_state = {
    "audio_branch":     model.model.audio_branch.state_dict(),
    "audio_transform":  model.model.audio_transform.state_dict(),
    "audio_projection": model.model.audio_projection.state_dict(),
    "logit_scale_a":    model.model.logit_scale_a.detach().cpu(),
}
torch.save(audio_state, AUDIO_CKPT_DIR / "audio_encoder_phase2.pt")

audio_meta = {
    "lora_r":        LORA_R,
    "lora_alpha":    LORA_ALPHA,
    "lora_dropout":  LORA_DROPOUT,
    "lr_audio":      LR_AUDIO,
    "epochs_audio":  EPOCHS_AUDIO,
    "audio_stages":  AUDIO_STAGES,
    "loss_per_epoch": audio_history,
}
with open(AUDIO_CKPT_DIR / "training_meta.json", "w") as f:
    json.dump(audio_meta, f, indent=2)

print(f"Phase-2 checkpoint saved → {AUDIO_CKPT_DIR}")

## Re-embed FMA with the fine-tuned audio encoder

In [ ]:
print("Re-embedding FMA tracks with Phase-2 audio encoder …")
model.eval()

# get_audio_embedding_from_filelist uses the updated audio branch internally
fma_embs_p2 = model.get_audio_embedding_from_filelist(fma_paths, use_tensor=False)
fma_embs_p2 = utils.l2_normalize(np.asarray(fma_embs_p2, dtype=np.float32))

print(f"Done — {len(fma_embs_p2)} FMA embeddings  shape={fma_embs_p2.shape}")

## Intra-genre cohesion: frozen vs fine-tuned audio encoder

In [ ]:
intra_sims_p2 = fma_utils.compute_intra_genre_cosine_similarity(
    genre_map, fma_paths, fma_embs_p2
)

print(f"FMA {FMA_SIZE.capitalize()} — Intra-genre cohesion: Phase 1 (frozen audio) vs Phase 2 (fine-tuned audio)")
print(f"{'Genre':<20} {'Phase 1':>10} {'Phase 2':>10} {'Delta':>8}")
print("-" * 52)
all_genres = sorted(set(intra_genre_sims) | set(intra_sims_p2))
for genre in all_genres:
    p1 = intra_genre_sims.get(genre, float("nan"))
    p2 = intra_sims_p2.get(genre, float("nan"))
    delta = p2 - p1
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "—")
    print(f"{genre:<20} {p1:>10.4f} {p2:>10.4f} {arrow}{abs(delta):>7.4f}")